In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance

In [ ]:
data = pd.read_excel("sample data.xlsx")
text_cols = [f"Q{i}_CH" for i in range(1, 34) if i != 22]
data["combined_text"] = (
    data[text_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)
targetvariable = "gender"
data_1 = data.loc[
    data[targetvariable].isin(["Man", "Woman"])
    & data["combined_text"].ne("")
].copy()

In [ ]:
train, test = train_test_split(data_1, test_size=0.20, random_state=806, stratify=data_1[targetvariable])
vectorizer = TfidfVectorizer(min_df=5, max_features=2000, stop_words="english")

X_train = vectorizer.fit_transform(train["combined_text"])
X_test = vectorizer.transform(test["combined_text"])

y_train = train[targetvariable]
y_test = test[targetvariable]

feature_names = vectorizer.get_feature_names_out()

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=806)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions, zero_division=0))

majority_class = y_train.mode()[0]
print("Baseline accuracy:", round((y_test == majority_class).mean(), 3))

In [ ]:
coefficients = pd.DataFrame({
    "word": feature_names,
    "coefficient": model.coef_[0]
})
coefficients["absolute_coefficient"] = coefficients["coefficient"].abs()
coefficients = coefficients.sort_values(
    "absolute_coefficient", ascending=False
)

print(f"\nPositive coefficients favor: {model.classes_[1]}")
print(f"Negative coefficients favor: {model.classes_[0]}")
print(coefficients.head(20).to_string(index=False))

top_coef = coefficients.head(20).sort_values("coefficient")

plt.figure(figsize=(9, 7))
plt.barh(
    top_coef["word"],
    top_coef["coefficient"],
    color=np.where(top_coef["coefficient"] > 0, "steelblue", "coral")
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel(f"Coefficient: positive favors {model.classes_[1]}")
plt.title("Top 20 Words by Absolute Logistic Regression Coefficient")
plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(
    model,
    X_test.toarray(),
    y_test,
    scoring="accuracy",
    n_repeats=10,
    random_state=806
)

importance = pd.DataFrame({
    "word": feature_names,
    "importance": perm.importances_mean,
    "SD": perm.importances_std
}).sort_values("importance", ascending=False)

print("\nPermutation importance:")
print(importance.head(20).to_string(index=False))

In [ ]:
top_perm = importance.head(20).sort_values("importance")

plt.figure(figsize=(9, 7))
plt.barh(
    top_perm["word"],
    top_perm["importance"],
    xerr=top_perm["SD"],
    color="steelblue"
)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Mean decrease in test accuracy after shuffling")
plt.title("Top 20 Words by Permutation Importance")
plt.tight_layout()
plt.show()